In [1]:
import os, json, gc
from typing import Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
# ===================== Konfiguration =====================
# -> Anpassen, falls nötig
MODEL_ID   = "Qwen/Qwen3-4B"              # HF-ID oder lokaler Pfad
TRAIN_JSON = "12B_trainingdata.json"
TEST_JSON  = "12B_golden_testdata.json"

WORKDIR    = "Output"
OUTDIR     = os.path.join(WORKDIR, "Output_neu", "qwen3_prune_run")
CKPT_FILE  = os.path.join(OUTDIR, "l0_state.pt")  # optional: Gate-Checkpoint

os.makedirs(OUTDIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if device.type == "cuda" else torch.float32  # A10/A100: bf16 ok

# Training
max_length   = 256
batch_size   = 2            # kleiner halten; wir akkumulieren Gradienten
num_epochs   = 3
GRAD_ACCUM   = 4            # -> effektiver Batch = batch_size * GRAD_ACCUM

# L0-Training (nur Gates werden gelernt)
lr              = 5e-4
grad_clip       = 1.0
lambda_l0_base  = 1e-3       # etwas höher als 5e-4 ist ok, aber nicht zu hoch
l0_warmup_steps = 2000
save_every_steps = 500

# Pruning-Strategie
SKIP_FIRST_LAYERS = 8        # frühe Layer unpruned lassen
MIN_SPARSITY      = 0.05     # bei frühen (geprunten) Layern
MAX_SPARSITY      = 0.20     # bei späten Layern (war 0.30 → zu aggressiv)
USE_LAYER_SCHEDULE = True    # linearer Anstieg von MIN->MAX über die Layer

# Optionales Nach-Finetuning NACH dem Schnitt
post_ft_steps = 2000         # Kurzes FT, kleine LR (im Code 5e-5)

LOG_EVERY = 50
print(f"Device: {device} | dtype: {dtype} | OUTDIR: {OUTDIR}")

Device: cuda | dtype: torch.bfloat16 | OUTDIR: Output/Output_neu/qwen3_prune_run


In [3]:
# ===================== Utils =====================
def count_trainable_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
def count_all_params(model):       return sum(p.numel() for p in model.parameters())

In [4]:
# ===================== Dataset (verbessert) =====================
class QADataset(Dataset):
    def __init__(self, path, tok, max_len=384, use_chat_template=True):
        with open(path, "r") as f:
            data = json.load(f)
        items = data.get("questions", data)

        self.tok = tok
        self.max_len = max_len
        self.use_chat_template = use_chat_template and hasattr(tok, "apply_chat_template")

        # Falls kein PAD definiert, nimm EOS
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token or "</s>"

        keep = []
        for it in items:
            q = (it.get("body") or it.get("question") or "").strip()
            if not q:
                continue

            # Antwort auswählen & normalisieren
            ans = it.get("ideal_answer")
            if ans is None or ans == "":
                ans = it.get("exact_answer")
            if ans is None or ans == "":
                continue
            if isinstance(ans, list):
                try:
                    ans = " ".join(map(str, ans))
                except Exception:
                    ans = str(ans)
            ans = str(ans).strip()
            if not ans:
                continue

            # Prompt bauen – bevorzugt via Chat-Template
            if self.use_chat_template:
                msgs = [
                    {"role": "user", "content": f"{q}"},
                    {"role": "assistant", "content": ""},  # wir lassen die Antwort generieren/labeln
                ]
                prompt = self.tok.apply_chat_template(
                    msgs,
                    tokenize=False,
                    add_generation_prompt=True,  # setzt das Assistant-Prelude korrekt
                )
                # Falls dein Template schon "Answer:" nicht enthält, kannst du optional ergänzen:
                # prompt += " Answer:"
            else:
                prompt = f"Question: {q}\nAnswer:"

            # Tokenisierung getrennt, damit wir präzise maskieren können
            enc_p = self.tok.encode(prompt, add_special_tokens=True)
            enc_a = self.tok.encode(ans, add_special_tokens=False)

            # EOS für die Antwort anhängen
            if self.tok.eos_token_id is not None:
                enc_a = enc_a + [self.tok.eos_token_id]

            # Wenn der Prompt bereits zu lang ist, skippen (sonst sind alle Labels -100)
            # oder wir kürzen den Prompt von links (besser: skippen, um saubere Supervision zu behalten)
            if len(enc_p) >= max_len - 8:   # 8 Tokens "Safety-Marge" für die Antwort
                # zu langer Prompt → wenig Learning-Signal; wir überspringen
                continue

            # So viel von der Antwort mitnehmen, wie reinpasst
            room = max_len - len(enc_p)
            enc_a = enc_a[:max(0, room)]

            input_ids = enc_p + enc_a
            labels    = [-100] * len(enc_p) + enc_a

            # Falls die Antwort komplett abgeschnitten wurde (keine supervisierten Tokens) → skippen
            if all(t == -100 for t in labels) or len(labels) == labels.count(-100):
                continue

            keep.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels":    torch.tensor(labels,    dtype=torch.long),
            })

        self.samples = keep
        print(f"Loaded {len(self.samples)} clean samples from {path} "
              f"(chat_template={'on' if self.use_chat_template else 'off'})")

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate(batch, pad_id, pad_to_multiple_of=8):
    # Ziel-Länge = maxlen, optional auf Vielfaches von 8 runden (besser für Tensor Cores)
    maxlen = max(len(x["input_ids"]) for x in batch)
    if pad_to_multiple_of and (maxlen % pad_to_multiple_of != 0):
        maxlen = ((maxlen + pad_to_multiple_of - 1) // pad_to_multiple_of) * pad_to_multiple_of

    input_ids, labels, attn = [], [], []
    for x in batch:
        pad = maxlen - len(x["input_ids"])
        ids = torch.cat([x["input_ids"], torch.full((pad,), pad_id, dtype=torch.long)])
        lbs = torch.cat([x["labels"],    torch.full((pad,), -100, dtype=torch.long)])
        am  = (ids != pad_id).long()
        input_ids.append(ids); labels.append(lbs); attn.append(am)

    return {
        "input_ids": torch.stack(input_ids),
        "labels":    torch.stack(labels),
        "attention_mask": torch.stack(attn),
    }

In [5]:
# ===================== Load Model/Tokenizer =====================
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1) Qwen lädt man am besten mit trust_remote_code
tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    trust_remote_code=True
)

# 2) PAD sauber setzen + in Model-Config spiegeln
if tok.pad_token is None:
    tok.pad_token = tok.eos_token or "</s>"
tok.padding_side = "right"  # fürs Training ok (wir maskieren ohnehin)

# 3) Datasets (falls du das Chat-Template nutzen willst: use_chat_template=True)
train_ds = QADataset(TRAIN_JSON, tok, max_len=max_length, use_chat_template=True)
test_ds  = QADataset(TEST_JSON,  tok, max_len=max_length, use_chat_template=True)

from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    num_workers=2, pin_memory=True,
    collate_fn=lambda b: collate(b, tok.pad_token_id)
)
test_loader  = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=2, pin_memory=True,
    collate_fn=lambda b: collate(b, tok.pad_token_id)
)

# 4) Modell laden
# dtype nur setzen, wenn es wirklich eine niedrigere Präzision ist
_load_dtype = dtype if dtype in (torch.float16, torch.bfloat16) else None
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=_load_dtype,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    # attn_implementation="flash_attention_2",  # <- optional, nur wenn installiert
)

# 5) Runtime-Details
model.config.use_cache = False
model.config.pad_token_id = tok.pad_token_id
model.config.eos_token_id = tok.eos_token_id

# Wir trainieren nur Gates → Basismodell einfrieren
for p in model.parameters():
    p.requires_grad = False

model.to(device)
print(f"Params (gesamt, vor Gates): {count_all_params(model):,}")

Loaded 5049 clean samples from 12B_trainingdata.json (chat_template=on)
Loaded 340 clean samples from 12B_golden_testdata.json (chat_template=on)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Params (gesamt, vor Gates): 4,022,468,096


In [6]:
# ===================== L0-Gates an down_proj =====================
import re

TARGET_UP   = "up_proj"
TARGET_GATE = "gate_proj"
TARGET_DOWN = "down_proj"

class GatedDownProj(nn.Module):
    """Wrapper um down_proj: lernt fp32-Gates (log_alpha) pro MLP-Kanal."""
    def __init__(self, base_linear: nn.Linear):
        super().__init__()
        self.base = base_linear
        inter_size = base_linear.in_features
        # Gates in fp32 halten; auf x.dtype nur im Forward casten
        self.log_alpha = nn.Parameter(torch.zeros(inter_size, dtype=torch.float32,
                                                  device=base_linear.weight.device))
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        gate = torch.sigmoid(self.log_alpha.to(x.dtype))
        return self.base(x * gate)

def _get_parent_and_name(root: nn.Module, full_name: str) -> Tuple[nn.Module, str]:
    parts = full_name.split("."); parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]

def _extract_layer_idx(name: str) -> int:
    """
    Versucht, eine Layer-Nummer aus Pfaden wie 'model.layers.12.mlp.down_proj' zu ziehen.
    Fällt auf -1 zurück, wenn nichts gefunden.
    """
    m = re.search(r"(layers|h|block|blocks)\.(\d+)", name)
    return int(m.group(2)) if m else -1

mlp_groups = []
for full_name, mod in list(model.named_modules()):
    if full_name.endswith(TARGET_DOWN) and isinstance(mod, nn.Linear):
        parent, down_name = _get_parent_and_name(model, full_name)
        up_mod   = getattr(parent, TARGET_UP,   None)
        gate_mod = getattr(parent, TARGET_GATE, None)
        if not (isinstance(up_mod, nn.Linear) and isinstance(gate_mod, nn.Linear)):
            continue

        # down_proj wrappen
        gated = GatedDownProj(mod)
        setattr(parent, down_name, gated)

        mlp_groups.append({
            "layer_idx": _extract_layer_idx(full_name),
            "name": full_name,
            "parent": parent,
            "up_name": TARGET_UP,
            "gate_name": TARGET_GATE,
            "down_name": down_name,                 # jetzt GatedDownProj
            "intermediate_size": mod.in_features,
            "hidden_size": mod.out_features
        })

# Gruppen nach Layer sortieren (für Skip/Schedule später)
mlp_groups.sort(key=lambda g: g["layer_idx"])

# Nur die Gates trainierbar machen & Optimizer strikt auf log_alpha setzen
for p in model.parameters():  # Sicherheitsgurt
    p.requires_grad = False
gate_params = [getattr(g["parent"], g["down_name"]).log_alpha for g in mlp_groups]
for gp in gate_params:
    gp.requires_grad = True

model.train()  # wir trainieren NUR die Gates

TOTAL_GATES = sum(gp.numel() for gp in gate_params)
print(f"MLP-Blöcke mit Gates: {len(mlp_groups)} | TOTAL_GATES: {TOTAL_GATES:,}")
print(f"Nur Gates trainierbar: {sum(p.numel() for p in gate_params):,} Parameter")

optimizer = torch.optim.AdamW(gate_params, lr=lr, weight_decay=0.0, eps=1e-8)
loss_fct = nn.CrossEntropyLoss(ignore_index=-100, reduction="sum")

MLP-Blöcke mit Gates: 36 | TOTAL_GATES: 350,208
Nur Gates trainierbar: 350,208 Parameter


In [7]:
# --------- Resume-Checkpoint (Gates + Optimizer) ----------
import os
import torch

CKPT_FILE = "checkpoint_gates.pt"

def save_ckpt(step, optimizer, path=CKPT_FILE):
    state = {
        "step": step,
        "optimizer": optimizer.state_dict(),
        "gates": [getattr(g["parent"], g["down_name"]).log_alpha.detach().cpu() for g in mlp_groups],
    }
    torch.save(state, path)
    print(f"[Checkpoint] gespeichert @ step={step}")

def load_ckpt(optimizer, path=CKPT_FILE):
    if not os.path.exists(path):
        print("[Checkpoint] nichts zum Laden gefunden")
        return 0
    state = torch.load(path, map_location="cpu")
    for g, loga in zip(mlp_groups, state["gates"]):
        getattr(g["parent"], g["down_name"]).log_alpha.data.copy_(
            loga.to(torch.float32).to(device)
        )
    try:
        optimizer.load_state_dict(state["optimizer"])
    except Exception as e:
        print(f"[Warnung] Optimizer-State konnte nicht geladen werden: {e}")
    step = int(state.get("step", 0))
    print(f"[Resume] geladen von {path} @ step={step}")
    return step

# Initiales Laden
global_step = load_ckpt(optimizer)

[Checkpoint] nichts zum Laden gefunden


In [8]:
import shutil
import os

# Cleanup nur auf das OUTDIR aus der Config anwenden
if os.path.exists(OUTDIR):
    print(f"[Cleanup] Entferne alten Output-Ordner: {OUTDIR}")
    shutil.rmtree(OUTDIR)

os.makedirs(OUTDIR, exist_ok=True)
print(f"[Setup] Neuer Output-Ordner erstellt: {OUTDIR}")

[Cleanup] Entferne alten Output-Ordner: Output/Output_neu/qwen3_prune_run
[Setup] Neuer Output-Ordner erstellt: Output/Output_neu/qwen3_prune_run


In [9]:
# ===================== Gates trainieren =====================

def evaluate(model, test_loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            labels    = batch["labels"].to(device)
            attn_mask = batch["attention_mask"].to(device)

            out = model(input_ids=input_ids, attention_mask=attn_mask)
            logits = out.logits[:, :-1, :].contiguous().to(torch.float32)
            shift_labels = labels[:, 1:].contiguous()

            num_toks = (shift_labels != -100).sum().item()
            if num_toks == 0:
                continue

            ce_sum = loss_fct(logits.view(-1, logits.size(-1)),
                              shift_labels.view(-1))
            total_loss += ce_sum.item()
            total_tokens += num_toks
    return total_loss / max(1, total_tokens)

for epoch in range(num_epochs):
    running, seen = 0.0, 0
    print(f"\n[Epoch {epoch+1}] startet – {len(train_loader)} Batches")
    model.train()
    for i, batch in enumerate(train_loader):
        global_step += 1

        input_ids = batch["input_ids"].to(device)
        labels    = batch["labels"].to(device)
        attn_mask = batch["attention_mask"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = out.logits
        shift_logits = logits[:, :-1, :].contiguous().to(torch.float32)
        shift_labels = labels[:, 1:].contiguous()

        num_toks = (shift_labels != -100).sum().item()
        if num_toks == 0:
            continue

        ce_sum = loss_fct(shift_logits.view(-1, shift_logits.size(-1)),
                          shift_labels.view(-1))
        ce_loss = ce_sum / float(num_toks)

        # L0-Term
        l0_sum = torch.zeros((), dtype=torch.float32, device=device)
        for g in mlp_groups:
            gd = getattr(g["parent"], g["down_name"])
            l0_sum = l0_sum + torch.sigmoid(gd.log_alpha).sum()
        l0_norm = l0_sum / float(TOTAL_GATES)

        warm_frac   = min(1.0, global_step / float(max(1, l0_warmup_steps)))
        lambda_curr = lambda_l0_base * warm_frac

        loss = ce_loss.to(dtype) + (lambda_curr * l0_norm).to(dtype)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], grad_clip)
        optimizer.step()

        running += float(loss.detach().to(torch.float32)); seen += 1
        if (i + 1) % LOG_EVERY == 0:
            avg = running / max(1, seen)
            print(f"Epoch {epoch+1} | {i+1}/{len(train_loader)} | avg loss={avg:.4f} "
                  f"| l0={l0_norm.detach().item():.4f} | λ={lambda_curr:.2e}")

        if global_step % save_every_steps == 0:
            save_ckpt(global_step, optimizer)   # ← richtig (2 Args + default path)
            print(f"[Checkpoint] saved @ step {global_step} -> {CKPT_FILE}")

    avg_train_loss = running / max(1, seen)
    val_loss = evaluate(model, test_loader)
    print(f"Epoch {epoch+1} done. Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")

# Letzten Gates-Stand sichern
save_ckpt(global_step, optimizer)        # ← richtig
print(f"[Checkpoint] final saved @ step {global_step}")


[Epoch 1] startet – 2525 Batches


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 1 | 50/2525 | avg loss=4.0788 | l0=0.5000 | λ=2.50e-05
Epoch 1 | 100/2525 | avg loss=3.6331 | l0=0.5000 | λ=5.00e-05
Epoch 1 | 150/2525 | avg loss=3.4446 | l0=0.5000 | λ=7.50e-05
Epoch 1 | 200/2525 | avg loss=3.3153 | l0=0.5001 | λ=1.00e-04
Epoch 1 | 250/2525 | avg loss=3.1752 | l0=0.5001 | λ=1.25e-04
Epoch 1 | 300/2525 | avg loss=3.0897 | l0=0.5001 | λ=1.50e-04
Epoch 1 | 350/2525 | avg loss=3.0165 | l0=0.5001 | λ=1.75e-04
Epoch 1 | 400/2525 | avg loss=2.9754 | l0=0.5002 | λ=2.00e-04
Epoch 1 | 450/2525 | avg loss=2.9337 | l0=0.5002 | λ=2.25e-04
Epoch 1 | 500/2525 | avg loss=2.8846 | l0=0.5002 | λ=2.50e-04
[Checkpoint] gespeichert @ step=500
[Checkpoint] saved @ step 500 -> checkpoint_gates.pt
Epoch 1 | 550/2525 | avg loss=2.8607 | l0=0.5002 | λ=2.75e-04
Epoch 1 | 600/2525 | avg loss=2.8069 | l0=0.5003 | λ=3.00e-04
Epoch 1 | 650/2525 | avg loss=2.7681 | l0=0.5003 | λ=3.25e-04
Epoch 1 | 700/2525 | avg loss=2.7295 | l0=0.5003 | λ=3.50e-04
Epoch 1 | 750/2525 | avg loss=2.7007 | l0=0.

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 1 done. Train Loss: 2.2617 | Val Loss: 1.6015

[Epoch 2] startet – 2525 Batches


...

Epoch 2 done. Train Loss: 1.8803 | Val Loss: 1.5181

[Epoch 3] startet – 2525 Batches


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 3 | 50/2525 | avg loss=1.8410 | l0=0.5023 | λ=1.00e-03
Epoch 3 | 100/2525 | avg loss=1.8354 | l0=0.5023 | λ=1.00e-03
Epoch 3 | 150/2525 | avg loss=1.8022 | l0=0.5024 | λ=1.00e-03
Epoch 3 | 200/2525 | avg loss=1.7924 | l0=0.5024 | λ=1.00e-03
Epoch 3 | 250/2525 | avg loss=1.7908 | l0=0.5024 | λ=1.00e-03
Epoch 3 | 300/2525 | avg loss=1.7926 | l0=0.5024 | λ=1.00e-03
Epoch 3 | 350/2525 | avg loss=1.7905 | l0=0.5024 | λ=1.00e-03
Epoch 3 | 400/2525 | avg loss=1.8074 | l0=0.5024 | λ=1.00e-03
Epoch 3 | 450/2525 | avg loss=1.8189 | l0=0.5024 | λ=1.00e-03
[Checkpoint] gespeichert @ step=5500
[Checkpoint] saved @ step 5500 -> checkpoint_gates.pt
Epoch 3 | 500/2525 | avg loss=1.8210 | l0=0.5025 | λ=1.00e-03
Epoch 3 | 550/2525 | avg loss=1.8119 | l0=0.5025 | λ=1.00e-03
Epoch 3 | 600/2525 | avg loss=1.8128 | l0=0.5025 | λ=1.00e-03
Epoch 3 | 650/2525 | avg loss=1.8078 | l0=0.5025 | λ=1.00e-03
Epoch 3 | 700/2525 | avg loss=1.8022 | l0=0.5025 | λ=1.00e-03
Epoch 3 | 750/2525 | avg loss=1.8020 | l0=

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 3 done. Train Loss: 1.7899 | Val Loss: 1.4806
[Checkpoint] gespeichert @ step=7575
[Checkpoint] final saved @ step 7575


In [11]:
# ===================== Physisches Pruning =====================
target_sparsity_mlp = 0.3   # 50% der MLP-Neuronen entfernen (anpassen nach Bedarf)

orig_inter = mlp_groups[0]["intermediate_size"]
keep_k     = max(1, int(round((1.0 - target_sparsity_mlp) * orig_inter)))
print(f"\n[Pruning] intermediate_size {orig_inter} -> {keep_k} "
      f"({int(100*(1 - keep_k/orig_inter))}% entfernt)")

class _Dummy: 
    pass

@torch.no_grad()
def prune_one_mlp_group(group, keep_idx_tensor: torch.Tensor):
    parent = group["parent"]
    up:   nn.Linear = getattr(parent, group["up_name"])
    gate: nn.Linear = getattr(parent, group["gate_name"])
    down_module     = getattr(parent, group["down_name"])
    assert isinstance(down_module, GatedDownProj)
    base_down: nn.Linear = down_module.base

    W_up,   b_up   = up.weight.data,   (up.bias.data   if up.bias   is not None else None)
    W_gate, b_gate = gate.weight.data, (gate.bias.data if gate.bias is not None else None)
    W_down, b_down = base_down.weight.data, (base_down.bias.data if base_down.bias is not None else None)

    dev, dt = W_up.device, W_up.dtype
    keep_k = keep_idx_tensor.numel()

    new_up   = nn.Linear(up.in_features,   keep_k, bias=(b_up is not None)).to(dev, dtype=dt)
    new_gate = nn.Linear(gate.in_features, keep_k, bias=(b_gate is not None)).to(dev, dtype=dt)
    new_down = nn.Linear(keep_k, base_down.out_features, bias=(b_down is not None)).to(dev, dtype=dt)

    new_up.weight.data.copy_(W_up[keep_idx_tensor, :])
    if b_up   is not None: new_up.bias.data.copy_(b_up[keep_idx_tensor])

    new_gate.weight.data.copy_(W_gate[keep_idx_tensor, :])
    if b_gate is not None: new_gate.bias.data.copy_(b_gate[keep_idx_tensor])

    new_down.weight.data.copy_(W_down[:, keep_idx_tensor])
    if b_down is not None: new_down.bias.data.copy_(b_down)

    setattr(parent, group["up_name"],   new_up)
    setattr(parent, group["gate_name"], new_gate)
    setattr(parent, group["down_name"], new_down)  # GatedDownProj -> plain Linear

with torch.no_grad():
    for g in mlp_groups:
        gd = getattr(g["parent"], g["down_name"])
        gates = torch.sigmoid(gd.log_alpha.detach().to(torch.float32)).cpu()
        keep_idx = torch.topk(gates, k=keep_k, largest=True).indices
        prune_one_mlp_group(g, keep_idx)

gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

# ===================== Config anpassen =====================
if hasattr(model.config, "intermediate_size"):
    model.config.intermediate_size = keep_k
elif hasattr(model.config, "hidden_size"):
    # Fallback – falls Qwen eine andere Namenskonvention nutzt
    model.config.hidden_size = keep_k
elif hasattr(model.config, "mlp_ratio"):
    # Manche Modelle definieren nur ein Verhältnis
    model.config.mlp_ratio = keep_k / model.config.hidden_size

print(f"Params (nach Pruning): {count_all_params(model):,}")


[Pruning] intermediate_size 9728 -> 6810 (29% entfernt)
Params (nach Pruning): 3,215,699,456


In [13]:
# ===================== Optional: kurzes Nach-Finetuning =====================
import math

if post_ft_steps > 0:
    print(f"\n[Post-FT] Starte kurzes Nach-Finetuning für {post_ft_steps} Schritte …")

    # 1) Nur MLP-Projektionen (up/gate/down) nach dem Pruning trainierbar machen
    for name, m in model.named_modules():
        if name.endswith(TARGET_UP) or name.endswith(TARGET_GATE) or name.endswith(TARGET_DOWN):
            for p in m.parameters():
                p.requires_grad = True
        else:
            for p in getattr(m, "parameters", lambda: [])():
                p.requires_grad = False

    # 2) Parameterliste exakt einsammeln
    ft_params = [p for p in model.parameters() if p.requires_grad]

    # 3) Optimizer (+ optional leichter Weight Decay)
    ft_lr = 5e-5
    optim2 = torch.optim.AdamW(ft_params, lr=ft_lr, betas=(0.9, 0.98), eps=1e-8, weight_decay=0.01)

    # 4) Scheduler: Cosine mit 5% Warmup
    warmup = max(10, int(0.05 * post_ft_steps))
    total_steps = post_ft_steps
    def lr_lambda(step):
        if step < warmup:
            return float(step) / float(max(1, warmup))
        # Cosine decay bis total_steps
        progress = (step - warmup) / float(max(1, total_steps - warmup))
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + math.cos(math.pi * (1.0 - progress)))
    sched2 = torch.optim.lr_scheduler.LambdaLR(optim2, lr_lambda=lr_lambda)

    # 5) Optional: Gradient Accumulation (eff. Batch = batch_size * GRAD_ACCUM)
    GRAD_ACCUM_FT = 4

    model.train()
    model.config.use_cache = False

    step = 0
    optim2.zero_grad(set_to_none=True)

    scaler_ctx = torch.autocast(device_type="cuda", dtype=dtype) if (device.type == "cuda" and dtype in (torch.float16, torch.bfloat16)) else contextlib.nullcontext()
    with scaler_ctx:
        for batch in train_loader:
            if step >= post_ft_steps:
                break

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            labels    = batch["labels"].to(device, non_blocking=True)
            attn_mask = batch["attention_mask"].to(device, non_blocking=True)

            out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss = out.loss

            # Accumulation
            loss = loss / GRAD_ACCUM_FT
            loss.backward()

            if (step + 1) % GRAD_ACCUM_FT == 0:
                # NaN/Inf-Guards (optional)
                for p in ft_params:
                    if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
                        p.grad = torch.nan_to_num(p.grad, nan=0.0, posinf=1.0, neginf=-1.0)

                torch.nn.utils.clip_grad_norm_(ft_params, 1.0)
                optim2.step()
                sched2.step()
                optim2.zero_grad(set_to_none=True)

            step += 1
            if step % 100 == 0:
                print(f"[Post-FT] step={step} | loss={float(loss.detach().to(torch.float32)):.4f} | lr={sched2.get_last_lr()[0]:.2e}")

    model.eval()
    print("[Post-FT] fertig.")


[Post-FT] Starte kurzes Nach-Finetuning für 2000 Schritte …


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[Post-FT] step=100 | loss=0.6251 | lr=1.25e-05
[Post-FT] step=200 | loss=0.7367 | lr=2.50e-05
[Post-FT] step=300 | loss=0.2752 | lr=3.75e-05
[Post-FT] step=400 | loss=0.5636 | lr=0.00e+00
[Post-FT] step=500 | loss=0.4703 | lr=2.14e-08
[Post-FT] step=600 | loss=0.4311 | lr=8.54e-08
[Post-FT] step=700 | loss=0.5608 | lr=1.92e-07
[Post-FT] step=800 | loss=0.5192 | lr=3.41e-07
[Post-FT] step=900 | loss=0.3860 | lr=5.32e-07
[Post-FT] step=1000 | loss=0.3775 | lr=7.65e-07
[Post-FT] step=1100 | loss=0.5401 | lr=1.04e-06
[Post-FT] step=1200 | loss=0.6328 | lr=1.35e-06
[Post-FT] step=1300 | loss=0.5386 | lr=1.71e-06
[Post-FT] step=1400 | loss=0.3820 | lr=2.11e-06
[Post-FT] step=1500 | loss=0.5067 | lr=2.54e-06
[Post-FT] step=1600 | loss=0.7150 | lr=3.01e-06
[Post-FT] step=1700 | loss=0.5585 | lr=3.52e-06
[Post-FT] step=1800 | loss=0.5126 | lr=4.07e-06
[Post-FT] step=1900 | loss=0.3376 | lr=4.65e-06
[Post-FT] step=2000 | loss=0.5224 | lr=5.27e-06
[Post-FT] fertig.


In [14]:
# ===================== Speichern =====================
PRUNED_DIR = os.path.join(OUTDIR, f"qwen3_pruned_mlp{keep_k}")
os.makedirs(PRUNED_DIR, exist_ok=True)

# 1) Modell + Tokenizer sichern
model.save_pretrained(PRUNED_DIR, safe_serialization=True)  # nutzt safetensors, wenn verfügbar
tok.save_pretrained(PRUNED_DIR)

# 2) Metadaten (z.B. Sparsity, Steps, Loss) speichern
meta = {
    "keep_k": keep_k,
    "target_sparsity_mlp": target_sparsity_mlp,
    "final_step": global_step,
    "post_ft_steps": post_ft_steps,
}
with open(os.path.join(PRUNED_DIR, "prune_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"\n[OK] Gepruntes Modell gespeichert nach: {PRUNED_DIR}")
print(f"[INFO] Metadaten: {meta}")


[OK] Gepruntes Modell gespeichert nach: Output/Output_neu/qwen3_prune_run/qwen3_pruned_mlp6810
[INFO] Metadaten: {'keep_k': 6810, 'target_sparsity_mlp': 0.3, 'final_step': 7575, 'post_ft_steps': 2000}
